# Workshop: Gemma from Scratch
## Notebook 5: Gated MLP (GeGLU)

**Estimated Time: 10 minutes**

In addition to attention, transformers use a Feed-Forward Network (FFN) to process each token. Gemma uses a **Gated Linear Unit (GLU)** variant. Instead of a simple `Linear -> ReLU -> Linear`, it uses three projections to create a gating mechanism.

### Learning Objectives:
1. Understand the SwiGLU/GeGLU architecture.
2. Implement the `gate_proj`, `up_proj`, and `down_proj` logic.
3. Compare GELU with standard ReLU.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

emb_dim = 128
hidden_dim = 512

### 1. Activation Functions: GELU

Gemma uses **GELU (Gaussian Error Linear Unit)**. Unlike ReLU, which is zero for all negative values, GELU is smooth and has a small "dip" into negative territory, which helps with gradient flow.

In [ ]:
x = torch.linspace(-5, 5, 100)
y_relu = F.relu(x)
y_gelu = F.gelu(x)

plt.plot(x.numpy(), y_relu.numpy(), label='ReLU')
plt.plot(x.numpy(), y_gelu.numpy(), label='GELU')
plt.legend()
plt.title("ReLU vs GELU")
plt.xlabel("Input")
plt.ylabel("Output")
plt.show()

### 2. The Gated Structure

The computation in a Gated MLP is:
$$Output = W_{down}(GELU(W_{gate} \cdot x) \odot (W_{up} \cdot x))$$

Where $\odot$ is element-wise multiplication.

In [ ]:
class GatedMLP(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.gate_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.up_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.down_proj = nn.Linear(d_hidden, d_in, bias=False)

    def forward(self, x):
        # 1. Compute gate and up projections
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        
        # 2. Apply activation and gating
        # In Gemma, we use GELU with tanh approximation
        activated_gate = F.gelu(gate, approximate="tanh")
        intermediate = activated_gate * up
        
        # 3. Project back down
        return self.down_proj(intermediate)

mlp = GatedMLP(emb_dim, hidden_dim)
sample_input = torch.randn(1, 10, emb_dim)
print("Output shape:", mlp(sample_input).shape)

### 3. Why Gating?

Gating allows the model to dynamically control the information flow for each token. The `gate_proj` decides *which* features from the `up_proj` are important for the current context.

### Exercise:
Compare the number of parameters in this `GatedMLP` vs a standard MLP (which only has one up projection and one down projection) with the same `hidden_dim`.

**Hints:**
1. Use `sum(p.numel() for p in model.parameters())` to count parameters.
2. A standard MLP has `self.up_proj = nn.Linear(d_in, d_hidden)` and `self.down_proj = nn.Linear(d_hidden, d_in)`.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

gated_params = count_parameters(mlp)
print(f"Gated MLP parameters: {gated_params}")

# Define and count a standard MLP
class StandardMLP(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.up_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.down_proj = nn.Linear(d_hidden, d_in, bias=False)
    def forward(self, x):
        return self.down_proj(F.relu(self.up_proj(x)))

standard_mlp = StandardMLP(emb_dim, hidden_dim)
standard_params = count_parameters(standard_mlp)
print(f"Standard MLP parameters: {standard_params}")

print(f"Difference: {gated_params - standard_params} parameters")

<details>
<summary><b>Click to see solution</b></summary>

The Gated MLP has approximately 50% more parameters than a standard MLP for the same hidden dimension because of the extra `gate_proj` layer.

```python
gated_params = sum(p.numel() for p in mlp.parameters())
standard_mlp = nn.Sequential(nn.Linear(emb_dim, hidden_dim, bias=False), nn.Linear(hidden_dim, emb_dim, bias=False))
standard_params = sum(p.numel() for p in standard_mlp.parameters())
```
</details>